# Week 4 Exercise - Frontier Code Generator

Author: BernardUdo

This notebook implements a modular code generator that converts Python snippets into optimized C++ using a frontier LLM model, then optionally compiles and benchmarks the output.

## Sections
- Configuration
- Prompt Templates
- LLM Code Generation
- File and Compilation Utilities
- Example Run

In [ ]:
import os
import re
import textwrap
import subprocess
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

In [ ]:
# -----------------------------
# Configuration Block
# -----------------------------
load_dotenv(override=True)

MODEL_NAME = "gpt-5"
OUTPUT_DIR = Path("generated")
CPP_FILENAME = "generated.cpp"
BINARY_NAME = "generated"

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise EnvironmentError("OPENAI_API_KEY is not set in your environment.")

client = OpenAI(api_key=OPENAI_API_KEY)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Using model: {MODEL_NAME}")

In [ ]:
# -----------------------------
# Prompt and Parsing Helpers
# -----------------------------
SYSTEM_PROMPT = textwrap.dedent(
    """
    You are an expert C++ performance engineer.
    Convert Python code into modern C++17 code optimized for runtime performance.

    Rules:
    1. Return only valid C++ code in a single fenced ```cpp block.
    2. Preserve the behavior of the original Python logic.
    3. Prefer efficient data structures and avoid unnecessary heap allocations.
    4. Include a simple main() demonstrating the function.
    """
).strip()


def build_user_prompt(python_code: str) -> str:
    return textwrap.dedent(
        f"""
        Convert this Python program into optimized C++17:

        ```python
        {python_code}
        ```
        """
    ).strip()


def extract_cpp_code(model_text: str) -> str:
    match = re.search(r"```cpp\s*(.*?)```", model_text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return model_text.strip()

In [ ]:
# -----------------------------
# Generation Utilities
# -----------------------------
def generate_cpp_from_python(python_code: str, model: str = MODEL_NAME) -> str:
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_prompt(python_code)},
        ],
    )
    raw_text = response.choices[0].message.content
    return extract_cpp_code(raw_text)


def write_cpp_file(cpp_code: str, output_dir: Path = OUTPUT_DIR, filename: str = CPP_FILENAME) -> Path:
    cpp_path = output_dir / filename
    cpp_path.write_text(cpp_code, encoding="utf-8")
    return cpp_path

In [ ]:
# -----------------------------
# Compile and Run Utilities
# -----------------------------
def compile_cpp(cpp_path: Path, binary_name: str = BINARY_NAME) -> Path:
    binary_path = cpp_path.parent / binary_name
    command = [
        "g++",
        "-O3",
        "-march=native",
        "-std=c++17",
        str(cpp_path),
        "-o",
        str(binary_path),
    ]
    subprocess.run(command, check=True, text=True, capture_output=True)
    return binary_path


def run_binary(binary_path: Path) -> str:
    result = subprocess.run([str(binary_path)], check=True, text=True, capture_output=True)
    return result.stdout.strip()

In [ ]:
# -----------------------------
# Example Run
# -----------------------------
python_source = textwrap.dedent(
    """
    def sum_of_squares(n: int) -> int:
        return sum(i * i for i in range(n + 1))

    if __name__ == "__main__":
        print(sum_of_squares(1000000))
    """
).strip()

cpp_code = generate_cpp_from_python(python_source)
cpp_file = write_cpp_file(cpp_code)
print(f"Generated C++ file: {cpp_file}")
print(cpp_code[:1200])

In [ ]:
# Optional compile + run
# Uncomment after verifying g++ is available on your machine.

# binary_file = compile_cpp(cpp_file)
# output = run_binary(binary_file)
# print(output)